# Reorganizing the BL media-list info to extract general metadata

The BL has provided us with some level of metdata in it's media list, which should be repurposed for our own metadata collection process, in particular the populating of the MediaSources ghseet document.


The information which is relevant to be fetched for each title/alias are:
- The NLP or list of NLPs
- The list of variant titles, coma separated with the years
- The project the data comes from (if porssible)
- The "country" (in UK terms) or publication area if available
- The publisher

The goal is to reformat the csv downloaded from the the sheet ["Compiled Confirmed Lists"](https://docs.google.com/spreadsheets/d/1srn9VpUZ9XkaImRxyCDzsLPfGBeUFPPecATsMhIKraU/edit?pli=1&gid=0#gid=0), and aggregate the information by alias to fit the columns of the [Impresso-MediaSources gheet](https://docs.google.com/spreadsheets/d/1jkW6cuINgT7SpuvJE7jVW4lpWiCypVuFiQOhuDZ_o1E/edit?gid=1371128556#gid=1371128556).

In [1]:
# Imports

import os
import pandas as pd
import numpy as np

### Read in the data, and remove excess or useless columns

In [ ]:
csv_medialist_path = '../data/api_metadata/06_BL-Media-List - Compiled Confirmed Lists.csv'

# read the data in, specifying some types

medialist_og_df = pd.read_csv(csv_medialist_path, header=1, index_col=0,
                              dtype={'NLP':'str', 'Start Year':'Int64', 'End Year':'Int64', 'Timespan (in years)':'Int64', 'Start year in Impresso local copy':'Int64', 'End year in Impresso local copy':'Int64'})
medialist_og_df.head()

,Working title,Variant Title,Unnamed: 3,NLP,Alias (in file-syst),Alias generated,Country,Start Year,End Year,Project Source,...,Notes about local copy,OCR Fomat,Deleted orginal copy (after was fully processed),# issues,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30
1,Aberdeen Press and Journal,Aberdeen Journal and General Advertiser,1,0000031,ANJO,ANJO,Scotland,1798,1876,JISC,...,"mostly BL-alias, but sometimes also abbyy-NLP ...",BL-Alias,"Yes, fully",4070.0,NaN,Total number of issues:,617623.0,NaN,Total number of Aliases:,372.0
2,Aberdeen Press and Journal,Aberdeen Weekly Journal and General Advertiser,2,0000032,ANJO,ANJO,Scotland,1876,1900,JISC,...,There were some small problems in the filenami...,BL-Alias,"Yes, fully",7301.0,NaN,OmniPage-NLP (Mets/Alto),173010.0,NaN,OmniPage-NLP (Mets/Alto),301.0
57,Age (London),The Age and Argus :,3,0002414,NaN,TALN,England,1825,1845,HMD,...,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",80.0,NaN,BL-Alias,200324.0,NaN,BL-Alias,36.0
58,Age 1852,The Age.,4,0003023,NaN,AGE52,England,1852,1853,HMD,...,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",37.0,NaN,ABBYY-NLP (w/ Mets),1436.0,NaN,ABBYY-NLP (w/ Mets),1.0
59,Agricultural Advertiser and Tenant-Farmers' Ad...,The Agricultural Advertiser and Tenant-Farmers...,5,0003031,NaN,AATA,England,1846,1846,HMD,...,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",31.0,NaN,ABBYY-Alias (w/ Mets),111080.0,NaN,ABBYY-Alias (w/ Mets),18.0


In [24]:
# remove all the unused or unecessary columns, as well as rows were the alias is undefined
medialist_df = medialist_og_df.copy()
medialist_df.reset_index(drop=True)
medialist_df = medialist_df.drop(columns=['Unnamed: 3', 'Alias (in file-syst)', 'Copyright (based on cutoff 1904)', 'Remark', 'BNL-impresso\nAcquisition Batch', 'Copy already shared with Impresso', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30'])
print(f"{len(medialist_df)} rows before removing ones without aliases")
medialist_df = medialist_df.dropna(subset='Alias generated')
print(f"{len(medialist_df)} rows after removing ones without aliases")
medialist_df.head()

669 rows before removing ones without aliases
648 rows after removing ones without aliases


,Working title,Variant Title,NLP,Alias generated,Country,Start Year,End Year,Project Source,Publisher,# Pages (estimate),Sheet of origin,Timespan (in years),Start year in Impresso local copy,End year in Impresso local copy,Notes about local copy,OCR Fomat,Deleted orginal copy (after was fully processed),# issues
1,Aberdeen Press and Journal,Aberdeen Journal and General Advertiser,0000031,ANJO,Scotland,1798,1876,JISC,D.C.Thomson & Co. Ltd,23 151,2023 Confirmed,79,1789,1876,"mostly BL-alias, but sometimes also abbyy-NLP ...",BL-Alias,"Yes, fully",4070.0
2,Aberdeen Press and Journal,Aberdeen Weekly Journal and General Advertiser,0000032,ANJO,Scotland,1876,1900,JISC,D.C.Thomson & Co. Ltd,55 751,2023 Confirmed,25,1877,1900,There were some small problems in the filenami...,BL-Alias,"Yes, fully",7301.0
57,Age (London),The Age and Argus :,0002414,TALN,England,1825,1845,HMD,Successor rightsholder unknown,8 950,2022 Confirmed,21,1843,1845,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",80.0
58,Age 1852,The Age.,0003023,AGE52,England,1852,1853,HMD,Successor rightsholder unknown,302,2022 Confirmed,2,1852,1853,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",37.0
59,Agricultural Advertiser and Tenant-Farmers' Ad...,The Agricultural Advertiser and Tenant-Farmers...,0003031,AATA,England,1846,1846,HMD,Successor rightsholder unknown,488,2022 Confirmed,1,1846,1846,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",31.0


### Grouping by Alias and creating the list of information we seek

First, prepare to group by NLP, by creating a string wti the variant title and years for each NLP

In [93]:
medialist_df['Other Titles'] = medialist_df.apply(lambda x: f"{x['Variant Title']} ({x['Start Year']}-{x['End Year']})" if not pd.isna(x['Start Year']) else f"{x['Variant Title']}", axis=1)

# create the per-NLP link to the British Library's newspaper archive
url = 'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/{nlp}'

medialist_df['url'] = medialist_df['NLP'].apply(lambda x: url.format(nlp=x))


medialist_df.head()

,Working title,Variant Title,NLP,Alias generated,Country,Start Year,End Year,Project Source,Publisher,# Pages (estimate),Sheet of origin,Timespan (in years),Start year in Impresso local copy,End year in Impresso local copy,Notes about local copy,OCR Fomat,Deleted orginal copy (after was fully processed),# issues,Other Titles,url
1,Aberdeen Press and Journal,Aberdeen Journal and General Advertiser,0000031,ANJO,Scotland,1798,1876,JISC,D.C.Thomson & Co. Ltd,23 151,2023 Confirmed,79,1789,1876,"mostly BL-alias, but sometimes also abbyy-NLP ...",BL-Alias,"Yes, fully",4070.0,Aberdeen Journal and General Advertiser (1798-...,https://www.britishnewspaperarchive.co.uk/sear...
2,Aberdeen Press and Journal,Aberdeen Weekly Journal and General Advertiser,0000032,ANJO,Scotland,1876,1900,JISC,D.C.Thomson & Co. Ltd,55 751,2023 Confirmed,25,1877,1900,There were some small problems in the filenami...,BL-Alias,"Yes, fully",7301.0,Aberdeen Weekly Journal and General Advertiser...,https://www.britishnewspaperarchive.co.uk/sear...
57,Age (London),The Age and Argus :,0002414,TALN,England,1825,1845,HMD,Successor rightsholder unknown,8 950,2022 Confirmed,21,1843,1845,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",80.0,The Age and Argus : (1825-1845),https://www.britishnewspaperarchive.co.uk/sear...
58,Age 1852,The Age.,0003023,AGE52,England,1852,1853,HMD,Successor rightsholder unknown,302,2022 Confirmed,2,1852,1853,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",37.0,The Age. (1852-1853),https://www.britishnewspaperarchive.co.uk/sear...
59,Agricultural Advertiser and Tenant-Farmers' Ad...,The Agricultural Advertiser and Tenant-Farmers...,0003031,AATA,England,1846,1846,HMD,Successor rightsholder unknown,488,2022 Confirmed,1,1846,1846,NaN,OmniPage-NLP (Mets/Alto),"Yes, fully",31.0,The Agricultural Advertiser and Tenant-Farmers...,https://www.britishnewspaperarchive.co.uk/sear...


In [94]:
medialist_df.values

array([['Aberdeen Press and Journal',
        'Aberdeen Journal and General Advertiser', '0000031', ...,
        4070.0, 'Aberdeen Journal and General Advertiser (1798-1876)',
        'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/0000031'],
       ['Aberdeen Press and Journal',
        'Aberdeen Weekly Journal and General Advertiser', '0000032', ...,
        7301.0,
        'Aberdeen Weekly Journal and General Advertiser (1876-1900)',
        'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/0000032'],
       ['Age (London)', 'The Age and Argus :', '0002414', ..., 80.0,
        'The Age and Argus : (1825-1845)',
        'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/0002414'],
       ...,
       ['York Herald', 'The York Herald', '0000499', ..., 5865.0,
        'The York Herald',
        'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/0000499'],
       ['York Herald', 'The York Herald', '0000500', 

In [95]:
import math

print(medialist_df['Notes about local copy'][59])
#t = np.nan()
if math.isnan(medialist_df['Notes about local copy'][59]):
    print('sey')


nan
sey


In [96]:
# create the agg function for the lists of categorical data, which might have repetitions

def agg_list_to_str(elems):
    return ', '.join(list(set(str(elem).strip() for elem in elems if str(elem).strip() != 'nan')))

In [97]:
340

340

In [98]:
metadata_per_alias = medialist_df.groupby('Alias generated').agg(
    {'Working title': list,
     'NLP': list,
     'Country': list,
     'Project Source': list,
     'Publisher': list,
     'Other Titles': list,
     'url': list,
     }
).map(lambda x: agg_list_to_str(x)).reset_index()


metadata_per_alias.head()

,Alias generated,Working title,NLP,Country,Project Source,Publisher,Other Titles,url
0,AATA,Agricultural Advertiser and Tenant-Farmers' Ad...,0003031,England,HMD,Successor rightsholder unknown,The Agricultural Advertiser and Tenant-Farmers...,https://www.britishnewspaperarchive.co.uk/sear...
1,AGE52,Age 1852,0003023,England,HMD,Successor rightsholder unknown,The Age. (1852-1853),https://www.britishnewspaperarchive.co.uk/sear...
2,AGMO,Anti-Gallican Monitor,"0002367, 0002364, 0002365, 0002366",England,HMD,Successor rightsholder unknown,"The Anti-Gallican Monitor. (1811-1811), The Br...",https://www.britishnewspaperarchive.co.uk/sear...
3,AHEC,Alston Herald and East Cumberland Advertiser,0003043,England,LWM,,"Alston Herald, and East Cumberland Advertiser....",https://www.britishnewspaperarchive.co.uk/sear...
4,ALBN,Albion,0003027,England,HMD,Successor rightsholder unknown,The Albion : (1852-1853),https://www.britishnewspaperarchive.co.uk/sear...


In [103]:
metadata_per_alias.iloc[13].values

array(['BDPO', 'Birmingham Daily Post', '0000033', 'England', '', '',
       'Birmingham Daily Post',
       'https://www.britishnewspaperarchive.co.uk/search/results/?IssueId=BL/0000033'],
      dtype=object)

In [104]:
# save the newly created dataframe to csv to be uploaded again to gdrive
out_metadata_path = '../data/api_metadata/metadata_per_alias.bl.csv'

metadata_per_alias.to_csv(out_metadata_path, index=False)